In [3]:
# 1. Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import joblib

In [4]:
# 2. Load dataset
df = pd.read_csv("UCLA_Master_Features-1-1-5.csv")  # exact filename use karo
print(df.shape)
df.head()
df.columns.tolist()

(109, 107)


['Unnamed: 0.1',
 'sr',
 'Unnamed: 0',
 'SUB_ID',
 'X',
 'subject',
 'SITE_ID',
 'FILE_ID',
 'DX_GROUP',
 'DSM_IV_TR',
 'AGE_AT_SCAN',
 'SEX',
 'HANDEDNESS_CATEGORY',
 'HANDEDNESS_SCORES',
 'FIQ',
 'VIQ',
 'PIQ',
 'FIQ_TEST_TYPE',
 'VIQ_TEST_TYPE',
 'PIQ_TEST_TYPE',
 'ADI_R_SOCIAL_TOTAL_A',
 'ADI_R_VERBAL_TOTAL_BV',
 'ADI_RRB_TOTAL_C',
 'ADI_R_ONSET_TOTAL_D',
 'ADI_R_RSRCH_RELIABLE',
 'ADOS_MODULE',
 'ADOS_TOTAL',
 'ADOS_COMM',
 'ADOS_SOCIAL',
 'ADOS_STEREO_BEHAV',
 'ADOS_RSRCH_RELIABLE',
 'ADOS_GOTHAM_SOCAFFECT',
 'ADOS_GOTHAM_RRB',
 'ADOS_GOTHAM_TOTAL',
 'ADOS_GOTHAM_SEVERITY',
 'SRS_VERSION',
 'SRS_RAW_TOTAL',
 'SRS_AWARENESS',
 'SRS_COGNITION',
 'SRS_COMMUNICATION',
 'SRS_MOTIVATION',
 'SRS_MANNERISMS',
 'SCQ_TOTAL',
 'AQ_TOTAL',
 'COMORBIDITY',
 'CURRENT_MED_STATUS',
 'MEDICATION_NAME',
 'OFF_STIMULANTS_AT_SCAN',
 'VINELAND_RECEPTIVE_V_SCALED',
 'VINELAND_EXPRESSIVE_V_SCALED',
 'VINELAND_WRITTEN_V_SCALED',
 'VINELAND_COMMUNICATION_STANDARD',
 'VINELAND_PERSONAL_V_SCALED',
 'VINELA

In [5]:
# 3. Basic cleaning

# -9999 ko treat as missing
df = df.replace(-9999, np.nan)

# ID / pure index type columns (adjust after dekh ke)
id_cols = ["SUBIDX", "subject", "SITEID", "FILEID", "srUnnamed 0"]  # jo bhi clearly ID lag rahe ho
id_cols = [c for c in id_cols if c in df.columns]
df = df.drop(columns=id_cols)

# Target presence check
assert "DX_GROUP" in df.columns, "DX_GROUP column missing"  # [file:2]

# Missing ratio dekhne ke liye
missing_ratio = df.isna().mean().sort_values(ascending=False)
# Bahut zyada missing (>0.5) columns hata do
high_missing = missing_ratio[missing_ratio > 0.5].index.tolist()
df = df.drop(columns=high_missing)
print("Dropped high-missing columns:", len(high_missing))

Dropped high-missing columns: 48


In [6]:
# Text-like columns drop (basic baseline)
text_cols = [c for c in df.columns if df[c].dtype == "object" and c != "DX_GROUP"]
df = df.drop(columns=text_cols)
print("Dropped text columns:", text_cols)

Dropped text columns: ['SITE_ID', 'FILE_ID', 'HANDEDNESS_CATEGORY', 'FIQ_TEST_TYPE', 'VIQ_TEST_TYPE', 'PIQ_TEST_TYPE', 'qc_rater_1', 'qc_anat_rater_2', 'qc_func_rater_2', 'qc_anat_rater_3', 'qc_func_rater_3']


In [7]:
df["DX_GROUP"].value_counts(dropna=False)

,count
DX_GROUP,
1,62
2,47


In [8]:
# 4. Target encode
df = df.dropna(subset=["DX_GROUP"])  # rows without diagnosis hatao
df["label"] = df["DX_GROUP"].map({1: 1, 2: 0})  # 1 = ASD, 0 = Non-ASD
df = df.drop(columns=["DX_GROUP"])

# Features / target split
X = df.drop(columns=["label"])
y = df["label"]

# Numeric columns list
num_cols = X.columns.tolist()

In [9]:
# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocessor: numeric columns + StandardScaler
numeric_transformer = Pipeline(steps=[
    ("imputer",  # simple mean imputation
     __import__("sklearn").impute.SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols)
    ]
)

In [10]:
# 6. Define models
log_reg = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

rf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        n_jobs=-1,
        class_weight="balanced",
        random_state=42
    ))
])

xgb = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    ))
])

models = {"log_reg": log_reg, "rf": rf, "xgb": xgb}

In [11]:
def evaluate_model(name, pipeline, X_train, X_test, y_train, y_test):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    print(f"=== {name} ===")
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    return pipeline

best_model = None
best_auc = -1
best_name = None

for name, pipe in models.items():
    trained = evaluate_model(name, pipe, X_train, X_test, y_train, y_test)
    auc = roc_auc_score(y_test, trained.predict_proba(X_test)[:, 1])
    if auc > best_auc:
        best_auc = auc
        best_model = trained
        best_name = name

print("Best model:", best_name, "with ROC-AUC:", best_auc)

=== log_reg ===
ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9
           1       1.00      1.00      1.00        13

    accuracy                           1.00        22
   macro avg       1.00      1.00      1.00        22
weighted avg       1.00      1.00      1.00        22

Confusion matrix:
 [[ 9  0]
 [ 0 13]]
=== rf ===
ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9
           1       1.00      1.00      1.00        13

    accuracy                           1.00        22
   macro avg       1.00      1.00      1.00        22
weighted avg       1.00      1.00      1.00        22

Confusion matrix:
 [[ 9  0]
 [ 0 13]]
=== xgb ===
ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9
           1       1.00      1.00      1.00        13

    accuracy                

In [12]:
# 7. Save best model for web app
joblib.dump(best_model, "asd_ucla_best_model.joblib")
joblib.dump(num_cols, "asd_ucla_feature_columns.joblib")

['asd_ucla_feature_columns.joblib']

In [13]:
import joblib
import pandas as pd

# 1) Model aur feature list load karo
model = joblib.load("asd_ucla_best_model.joblib")
feature_cols = joblib.load("asd_ucla_feature_columns.joblib")

# 2) Poora data dobara load karo (same CSV)
df_all = pd.read_csv("UCLA_Master_Features-1-1-5.csv")
df_all = df_all.replace(-9999, np.nan)

id_like = ["srUnnamed 0", "SUBIDX", "subject", "SITEID", "FILEID"]
id_like = [c for c in id_like if c in df_all.columns]
df_all = df_all.drop(columns=id_like)

df_all = df_all.dropna(subset=["DX_GROUP"])
df_all["label"] = df_all["DX_GROUP"].map({1: 1, 2: 0})
df_all = df_all.drop(columns=["DX_GROUP"])

text_cols = [c for c in df_all.columns if df_all[c].dtype == "object" and c != "label"]
df_all = df_all.drop(columns=text_cols)

X_all = df_all[feature_cols]
y_all = df_all["label"]

# 3) Model se prediction
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

y_pred = model.predict(X_all)
y_proba = model.predict_proba(X_all)[:, 1]

print("Full‑data ROC‑AUC:", roc_auc_score(y_all, y_proba))
print(classification_report(y_all, y_pred))
print(confusion_matrix(y_all, y_pred))

Full‑data ROC‑AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        47
           1       1.00      1.00      1.00        62

    accuracy                           1.00       109
   macro avg       1.00      1.00      1.00       109
weighted avg       1.00      1.00      1.00       109

[[47  0]
 [ 0 62]]


In [14]:
# ek sample sample ka manual test (web app jaise)
# Kisi ek row ko uthao (for example index 0) aur dekhो model kya predict karta hai:

sample = X_all.iloc[[0]]   # double bracket to keep DataFrame
true_label = int(y_all.iloc[0])

pred = model.predict(sample)[0]
proba = model.predict_proba(sample)[0, 1]

print("True label:", true_label, "(1=ASD, 0=Non-ASD)")
print("Predicted label:", pred)
print("ASD probability:", proba)

True label: 1 (1=ASD, 0=Non-ASD)
Predicted label: 1
ASD probability: 0.9952362665278508


In [15]:
# Train–test split based testing (recommended) for web app
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import joblib
import pandas as pd
import numpy as np

# 1) Data dubara tayyar karo (same steps)
df = pd.read_csv("UCLA_Master_Features-1-1-5.csv")
df = df.replace(-9999, np.nan)

id_like = ["srUnnamed 0", "SUBIDX", "subject", "SITEID", "FILEID"]
id_like = [c for c in id_like if c in df.columns]
df = df.drop(columns=id_like)

df = df.dropna(subset=["DX_GROUP"])
df["label"] = df["DX_GROUP"].map({1: 1, 2: 0})
df = df.drop(columns=["DX_GROUP"])
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

text_cols = [c for c in df.columns if df[c].dtype == "object" and c != "label"]
df = df.drop(columns=text_cols)

X = df.drop(columns=["label"])
y = df["label"]
num_cols = X.columns.tolist()

# 2) Train–test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)

# 3) Saved model load karo
model = joblib.load("asd_ucla_best_model.joblib")

# 4) Sirf TEST set par evaluation
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("TEST ROC-AUC:", roc_auc_score(y_test, y_proba))
print("TEST classification report:\n", classification_report(y_test, y_pred))
print("TEST confusion matrix:\n", confusion_matrix(y_test, y_pred))

Train size: (87, 89) Test size: (22, 89)
TEST ROC-AUC: 1.0
TEST classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         9
           1       1.00      1.00      1.00        13

    accuracy                           1.00        22
   macro avg       1.00      1.00      1.00        22
weighted avg       1.00      1.00      1.00        22

TEST confusion matrix:
 [[ 9  0]
 [ 0 13]]


In [16]:
!pip install streamlit joblib xgboost scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 109.2 MB/s eta 0:00:00


In [17]:
!pip install pyngrok --quiet
from pyngrok import ngrok

In [18]:
NGROK_AUTH_TOKEN = "yahan_apna_token_paste_karo"  # apna token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [19]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "yaha_real_token_paste_karo"  # poora token string
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [20]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3CAiWGEOpoYddohyVC7BGfEeJX5_3vhRTujQUiMGE52G13RgS"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [21]:
import subprocess, time

proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"]
)

time.sleep(5)
public_url = ngrok.connect(8501, "http")
print("App URL:", public_url)

App URL: NgrokTunnel: "https://eleven-confirm-sustained.ngrok-free.dev" -> "http://localhost:8501"
